# Notebook 4: Structural Alignment and Active Site Extraction

**Pipeline step:** Superimpose all 17 MetRS and 17 cupin Boltz2 predictions onto a reference structure, then map active site residues onto each enzyme.

**Overview:** Once we have 3D structures, we can ask: which residues line the active site in each HS enzyme? To answer this, we align every structure to a single reference (PyrN) and use the alignment to identify corresponding residues. This notebook teaches structural alignment, RMSD, the Biopython PDB module, and the logic of `analyse_metrs.py` and `analyse_cupin.py`.

---

## Learning objectives

- Understand why structural alignment is more informative than sequence alignment for divergent proteins
- Know what RMSD is and how to interpret it
- Navigate the Biopython PDB hierarchy: Structure, Model, Chain, Residue, Atom
- Parse CIF files and access coordinates
- Understand global vs local pairwise sequence alignment (Needleman-Wunsch vs Smith-Waterman)
- Use Biopython Superimposer to align structures and compute RMSD
- Understand what defines the MetRS and cupin active sites in HS enzymes
- Read and interpret the output active site CSV files
- Visualise the active site data as a heatmap

---

## 1. Structural vs sequence alignment

When comparing proteins, sequence alignment finds the best way to line up their amino acid strings. It works well for closely related proteins. However, protein structures evolve more slowly than sequences: two MetRS-fold enzymes can share the same active site geometry even at 25% sequence identity — a level where sequence alignment becomes unreliable.

**Structural alignment** compares the 3D coordinates of matched atoms, not the amino acid letters. This is more informative for:
- Identifying equivalent residues across divergent homologues
- Mapping active site positions conserved in structure but not sequence
- Quantifying how similar two folds are

In this pipeline we use a hybrid approach:
1. **Sequence alignment** (Needleman-Wunsch) to find which residue in enzyme X corresponds to which residue in PyrN
2. **Structural superimposition** (Biopython Superimposer) to compute RMSD after applying that correspondence

This works well when there is enough sequence similarity to get a good alignment (~30%+). For the MetRS domain, the HVGH catalytic motif provides a reliable anchor.

> **Key concept:** We use PyrN (a characterised HS from *S. candidus*) as the reference because it has an experimentally determined 3D structure (PDB: see `PyrN.pdb`) and its active site residues have been mapped biochemically (Hao 2026).

---

## 2. RMSD

**RMSD (Root Mean Square Deviation)** is the standard measure of structural similarity. After superimposing two structures to minimise it:

$$\text{RMSD} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} d_i^2}$$

where $d_i$ is the distance between matched atom pair $i$ after optimal superimposition.

We calculate RMSD over matched Cα atoms (one per residue — the backbone carbon).

**Interpretation for protein domains:**

| RMSD (Å) | Structural similarity |
|---|---|
| < 1 | Essentially identical (same crystal form or very close homologue) |
| 1 – 2 | Very similar fold; minor loop differences |
| 2 – 4 | Same fold, moderate divergence |
| > 4 | Distant relatives or different folds |

In the AAR-COMP-012 results, MetRS domain RMSDs range from ~2.7 to ~5.0 Å — reflecting the expected diversity within this enzyme family.

---

## 3. Biopython PDB module

Biopython provides `Bio.PDB` for reading and manipulating 3D structures. The hierarchy is:

```
Structure
  └── Model [0]            # one entry per MODEL record (usually just one)
        └── Chain ["A"]    # one per chain identifier
              └── Residue  # one per amino acid (or HETATM ligand)
                    └── Atom  # one per heavy atom
                          └── coordinates (Vector)
```

Residue `.id` is a tuple: `(' ', 138, ' ')` means standard residue at sequence position 138.

In [ ]:
# ── Parse PyrN.pdb and inspect chain A ────────────────────────────────
from Bio.PDB import PDBParser, MMCIFParser, Superimposer

PYRN_PDB = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step4/PyrN.pdb"

# PDBParser for .pdb files; MMCIFParser for .cif files
parser    = PDBParser(QUIET=True)
structure = parser.get_structure("PyrN", PYRN_PDB)
chain     = structure[0]["A"]     # Model 0, Chain A

# Filter to standard amino acid residues (exclude water, zinc, etc.)
residues = [r for r in chain if r.id[0] == " "]

# Build the sequence from residue names
AA3TO1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
}

sequence = "".join(AA3TO1.get(r.resname, "X") for r in residues)

print("PyrN chain A:")
print("  {} residues".format(len(residues)))
print("  Residue numbers: {} to {}".format(residues[0].id[1], residues[-1].id[1]))
print("  Sequence:")
# Print in blocks of 60
for i in range(0, len(sequence), 60):
    print("    {}".format(sequence[i:i+60]))

In [ ]:
# ── Identify AMP-site residues in PyrN ────────────────────────────────
# From Hao 2026, the AMP-binding site in PyrN (PDB numbering) is:

# (PDB residue number, paper label)
AMP_SITE = [
    (19, "S138"), (20, "F139"), (21, "P140"), (22, "T141"),
    (60, "V179"), (63, "Q182"),
    (265, "A384"), (268, "L387"), (271, "R390"),
    (302, "N421"), (306, "R425"),
]

# Note: PDB numbering = paper numbering - 119
# (The MetRS domain starts at residue 131 of full-length PyrN;
#  Biopython PDB residue numbers start from 1 within the chain)

print("AMP-binding site in PyrN crystal structure:")
print("{:<10} {:<8} {:<6}".format("PDB res#", "Label", "AA"))
print("-" * 28)

# Build a dict: PDB residue number → residue object
res_by_num = {r.id[1]: r for r in residues}

for pdb_num, label in AMP_SITE:
    res = res_by_num.get(pdb_num)
    aa  = AA3TO1.get(res.resname, "?") if res else "missing"
    print("{:<10} {:<8} {:<6}".format(pdb_num, label, aa))

---

## 4. Pairwise sequence alignment

Before superimposing two structures, we need to know which residue in enzyme X corresponds to which residue in PyrN. This is determined by **pairwise sequence alignment**.

**Global alignment (Needleman-Wunsch):** Aligns the full length of both sequences. Best when both sequences are roughly the same length and are homologous throughout (as here — we are comparing MetRS domain to MetRS domain).

**Local alignment (Smith-Waterman):** Finds the best-matching sub-region. Better when one sequence is much longer, or you only care about a conserved region.

**Scoring parameters used in the pipeline:**
- Match: +2 (reward for identical residues)
- Mismatch: -1 (penalty for different residues)
- Gap open: -10 (strong penalty to discourage gaps)
- Gap extend: -0.5 (small penalty for each additional gap position)

The strong gap open penalty ensures we get few, biologically meaningful insertions/deletions rather than many small gaps.

> **Key concept:** The alignment gives us a list of `(ref_index, target_index)` pairs — matched positions between PyrN and the enzyme we are analysing. We use those pairs to assemble lists of Cα atoms for superimposition, and also to identify which residue in the enzyme aligns to each active site position in PyrN.

In [ ]:
# ── Demonstrate pairwise alignment with Biopython ─────────────────────
from Bio.Align import PairwiseAligner

# Two short example sequences
seq_ref = "MHVKTLDRWE"    # made-up reference
seq_tgt = "MHVKTL-RWE"    # target with a gap (dash here is for illustration only)
seq_tgt = "MHVKTLRWE"     # actual query — 9 residues

aligner = PairwiseAligner()
aligner.mode             = "global"    # Needleman-Wunsch
aligner.match_score      = 2
aligner.mismatch_score   = -1
aligner.open_gap_score   = -10
aligner.extend_gap_score = -0.5

# Get the best alignment
aln = next(iter(aligner.align(seq_ref, seq_tgt)))
print("Score: {}".format(aln.score))
print()
print(aln)

In [ ]:
# ── Extract matched residue index pairs from an alignment ──────────────
# aln.aligned[0] gives the matched blocks in the reference as (start, end) slices
# aln.aligned[1] gives the matched blocks in the target

pairs = []
for (rs, re), (ts, te) in zip(aln.aligned[0], aln.aligned[1]):
    for d in range(re - rs):     # iterate over each matched position in the block
        pairs.append((rs + d, ts + d))   # (ref_index, target_index)

print("Matched index pairs (ref_idx, tgt_idx):")
for ref_i, tgt_i in pairs:
    print("  ref[{}]={} <-> tgt[{}]={}".format(
        ref_i, seq_ref[ref_i], tgt_i, seq_tgt[tgt_i]))

---

## 5. Biopython Superimposer

`Bio.PDB.Superimposer` takes two equal-length lists of `Atom` objects (reference atoms and target atoms), finds the optimal rotation and translation matrix that minimises the RMSD between them, and applies that transformation to the target atoms.

The key steps:
1. Collect matched Cα atoms from reference and target using the alignment pairs
2. Call `sup.set_atoms(ref_atoms, tgt_atoms)` — this computes the rotation matrix
3. Read `sup.rms` — the RMSD after superimposition
4. Optionally call `sup.apply(structure)` to rotate the target structure in place

In [ ]:
# ── Full align-and-superimpose function from analyse_metrs.py ─────────
from Bio.PDB import Superimposer

def get_ca_residues(chain):
    """Return list of standard residues that have a Cα atom."""
    return [r for r in chain if r.id[0] == " " and "CA" in r]


def get_seq(residues):
    """Convert a list of residue objects to a single-letter sequence string."""
    return "".join(AA3TO1.get(r.resname, "X") for r in residues)


def align_and_superimpose(ref_residues, tgt_residues):
    """
    1. Global pairwise sequence alignment to find matched positions.
    2. Extract matched Cα atoms.
    3. Superimpose and return RMSD plus index pairs.
    """
    aligner = PairwiseAligner()
    aligner.mode             = "global"
    aligner.match_score      = 2
    aligner.mismatch_score   = -1
    aligner.open_gap_score   = -10
    aligner.extend_gap_score = -0.5

    # Align sequences derived from the 3D structures
    aln   = next(iter(aligner.align(get_seq(ref_residues), get_seq(tgt_residues))))

    # Extract matched index pairs
    pairs = []
    for (rs, re), (ts, te) in zip(aln.aligned[0], aln.aligned[1]):
        for d in range(re - rs):
            pairs.append((rs + d, ts + d))

    # Collect Cα atoms for the matched pairs
    ref_atoms = [ref_residues[i]["CA"] for i, _ in pairs]
    tgt_atoms = [tgt_residues[j]["CA"] for _, j in pairs]

    # Superimpose: find rotation/translation that minimises RMSD
    sup = Superimposer()
    sup.set_atoms(ref_atoms, tgt_atoms)   # ref is fixed; tgt is rotated

    return sup.rms, pairs


print("align_and_superimpose() defined.")

In [ ]:
# ── Test superimposition: PyrN vs first available MetRS prediction ─────
import glob

metrs_cifs = sorted(glob.glob(
    "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step3/metrs/"
    "boltz_results_*/predictions/*_metrs/*_model_0.cif"
))

if metrs_cifs:
    cif_path = metrs_cifs[0]
    enzyme   = cif_path.split("/")[-2].replace("_metrs", "")
    print("Testing with: {}".format(enzyme))

    # Load the Boltz2 prediction
    cif_parser   = MMCIFParser(QUIET=True)
    tgt_struct   = cif_parser.get_structure("tgt", cif_path)
    tgt_residues = get_ca_residues(tgt_struct[0]["A"])

    # Load PyrN reference
    ref_residues = get_ca_residues(PDBParser(QUIET=True).get_structure("PyrN", PYRN_PDB)[0]["A"])

    rmsd, pairs = align_and_superimpose(ref_residues, tgt_residues)

    print("RMSD: {:.3f} A".format(rmsd))
    print("Matched pairs: {}".format(len(pairs)))
else:
    print("No MetRS CIF files found. Run SLURM jobs first.")

---

## 6. Active site definitions

### MetRS active site (Hao 2026)

Hao et al. (2026) characterised the active site of PyrN MetRS domain biochemically. Two functional subsites were identified:

- **AMP-binding site** (11 positions): S138, F139, P140, T141, V179, Q182, A384, L387, R390, N421, R425
  - S138 and T141 are fully conserved across all HS enzymes — essential hydroxyl groups for AMP recognition
- **N-OH substrate site** (6 positions): N143, E284, E286, D420, N421, L424
  - These residues interact with the N-hydroxylamine intermediate unique to hydrazine biosynthesis

### Cupin active site (ZN proximity)

For the cupin domain, there is no published biochemical active site map. Instead, `analyse_cupin.py` defines the active site empirically: all residues with any heavy atom within **5 Å of the zinc ion** in the best PyrN cupin Boltz2 model. This gives ~5 residues including the classic HxHxE zinc coordination tetrad typical of cupin superfamily members.

The cupin active site columns in `cupin_active_site.csv` are labelled by the PyrN residue identity and position: e.g. `H11`, `H13`, `E17`, `W19`, `H51`.

In [ ]:
# ── Read the MetRS active site results CSV ────────────────────────────
import pandas as pd

METRS_CSV = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step4/results/metrs_active_site.csv"

metrs_df = pd.read_csv(METRS_CSV)

print("Shape: {} rows x {} columns".format(metrs_df.shape[0], metrs_df.shape[1]))
print()
print("Columns:")
for col in metrs_df.columns:
    print("  {}".format(col))
print()
# Show first few rows — display only the metadata + AMP site columns
amp_cols = ["S138", "F139", "P140", "T141", "V179", "Q182",
            "A384", "L387", "R390", "N421", "R425"]
display_cols = ["Enzyme", "RMSD", "Confidence"] + amp_cols
print(metrs_df[display_cols].to_string(index=False))

In [ ]:
# ── Read the cupin active site CSV ────────────────────────────────────
CUPIN_CSV = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step4/results/cupin_active_site.csv"

cupin_df = pd.read_csv(CUPIN_CSV)

print("Cupin active site columns:")
print(cupin_df.columns.tolist())
print()
print(cupin_df.to_string(index=False))

In [ ]:
# ── Find the enzyme with the highest RMSD ─────────────────────────────
most_divergent = metrs_df.loc[metrs_df["RMSD"].idxmax()]

print("MetRS enzyme with highest RMSD:")
print("  Enzyme:     {}".format(most_divergent["Enzyme"]))
print("  RMSD:       {:.3f} A".format(most_divergent["RMSD"]))
print("  Confidence: {:.3f}".format(most_divergent["Confidence"]))
print()

# Show its active site residues vs PyrN
pyrn_row = metrs_df[metrs_df["Enzyme"] == "PyrN_crystal"].iloc[0]

print("{:<10} {:<10} {}".format("Position", "PyrN", most_divergent["Enzyme"]))
print("-" * 30)
for col in amp_cols:
    pyrn_aa = pyrn_row[col]
    tgt_aa  = most_divergent[col]
    flag    = "  <-- substitution" if pyrn_aa != tgt_aa else ""
    print("{:<10} {:<10} {}{}".format(col, pyrn_aa, tgt_aa, flag))

---

## 7. Visualising active site conservation — heatmap

A heatmap is a matrix plot where each cell's colour encodes a value. Here we encode whether each enzyme has the same amino acid as PyrN (1 = conserved, 0 = different). This gives an instant view of which positions are conserved and which are variable.

In [ ]:
# ── Conservation heatmap for MetRS AMP site ───────────────────────────
import matplotlib
matplotlib.use("Agg")   # non-interactive backend (needed on HPC)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Build a binary conservation matrix:
# 1 if the residue matches PyrN, 0 if it is different

# Exclude the PyrN_crystal row itself (it is always 1)
compare_df = metrs_df[metrs_df["Enzyme"] != "PyrN_crystal"].copy()

# PyrN reference amino acids for each AMP site position
pyrn_ref = {col: pyrn_row[col] for col in amp_cols}

# Build matrix: rows = enzymes, columns = positions
matrix = []
enzyme_labels = []

for _, row in compare_df.iterrows():
    row_vals = []
    for col in amp_cols:
        # 1 = same as PyrN, 0 = different
        row_vals.append(1 if row[col] == pyrn_ref[col] else 0)
    matrix.append(row_vals)
    enzyme_labels.append(row["Enzyme"])

matrix = np.array(matrix, dtype=float)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    matrix,
    annot=True,            # show values in each cell
    fmt=".0f",             # integer format
    cmap="YlGn",           # yellow = 0 (different), green = 1 (conserved)
    xticklabels=amp_cols,  # position labels on x-axis
    yticklabels=enzyme_labels,  # enzyme names on y-axis
    linewidths=0.4,
    ax=ax,
)
ax.set_title("MetRS AMP-site conservation (1=same as PyrN, 0=different)", pad=12)
ax.set_xlabel("Active site position (Hao 2026 numbering)")
ax.set_ylabel("Enzyme")
plt.tight_layout()
plt.savefig("/tmp/metrs_conservation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved to /tmp/metrs_conservation_heatmap.png")

---

## Summary

In this notebook you:
- Understood why structural alignment is used for divergent HS enzymes
- Learned what RMSD is and how to interpret values in the 1–5 Å range
- Navigated the Biopython PDB hierarchy and accessed coordinates from PyrN.pdb
- Used `PairwiseAligner` for global Needleman-Wunsch alignment to match residues
- Used `Superimposer` to compute RMSD between PyrN and Boltz2 models
- Explored the MetRS and cupin active site CSVs from Step 4
- Generated a conservation heatmap to visualise variation across enzymes

**Next:** Notebook 5 will convert the active site CSVs into publication-quality sequence logos.

---

## Try it yourself

**Exercise 1:** Parse `PyrN.pdb` (chain A) using Biopython, print the full sequence, and count the total number of residues.

```python
# Hint: use PDBParser, structure[0]['A'], and filter for r.id[0] == ' '
# your code here
```

**Exercise 2:** Load `metrs_active_site.csv` with pandas and print the enzyme name and RMSD for the five enzymes with the highest RMSD values.

```python
# Hint: df.sort_values("RMSD", ascending=False).head(5)
# your code here
```

**Exercise 3:** For each column (active site position) in `metrs_active_site.csv`, calculate how many unique amino acids appear across all enzymes. Print the position and its count — this tells you which positions are most variable.

```python
# Hint: for col in amp_cols: df[col].nunique()
# your code here
```